# Лабораторная работа 3.5. Численное интегрирование

**Вариант 7.** Вычислить интеграл

$$\int_{-2}^{2}\frac{dx}{3x^2+4x+2}$$

методами левых, правых и средних прямоугольников, трапеций и Симпсона с шагами
$h_1=1.0$ и $h_2=0.5$; уточнить результаты методом Рунге–Ромберга и сравнить с точным значением.

In [1]:
import json
import math
import os

import numpy as np
import pandas as pd

DATA_DIR = "data"

### Вспомогательные функции: чтение данных и оформление вывода

In [2]:
def read_data(name: str) -> dict:
    """Читает JSON-файл с исходными данными."""
    path = os.path.join(DATA_DIR, name)
    if not os.path.exists(path):
        raise FileNotFoundError(f"файл '{path}' не найден: положите папку data рядом с ноутбуком")
    with open(path, encoding="utf-8") as file:
        return json.load(file)


def vector(data: dict, key: str, length: int | None = None) -> np.ndarray:
    """Читает вектор вещественных чисел по ключу и проверяет его длину."""
    if key not in data:
        raise ValueError(f"в исходных данных нет ключа '{key}'")
    values = np.array([float(value) for value in data[key]])
    if length is not None and len(values) != length:
        raise ValueError(f"'{key}': {len(values)} элементов вместо {length}")
    return values


def scalar(data: dict, key: str) -> float:
    """Читает одно вещественное число по ключу."""
    if key not in data:
        raise ValueError(f"в исходных данных нет ключа '{key}'")
    return float(data[key])


def number(value: float, precision: int = 6) -> str:
    """Число в едином формате; машинный нуль печатается как 0."""
    value = float(value)
    if abs(value) < 0.5 * 10 ** (-precision):
        value = 0.0
    return f"{value:.{precision}f}"


def table(rows: list[list], headers: list[str], title: str | None = None) -> None:
    """Печатает таблицу (pandas используется только для оформления)."""
    if title:
        print(title)
    display(pd.DataFrame(rows, columns=headers).style.hide(axis="index"))

### Реализация методов

In [3]:
def f(x: float) -> float:
    """Подынтегральная функция варианта."""
    return 1.0 / (3.0 * x * x + 4.0 * x + 2.0)


def antiderivative(x: float) -> float:
    """Первообразная: знаменатель не имеет корней, интеграл выражается через арктангенс."""
    return math.atan((3.0 * x + 2.0) / math.sqrt(2.0)) / math.sqrt(2.0)


def nodes_of(left: float, right: float, step: float) -> np.ndarray:
    """Равномерная сетка; шаг должен укладываться целое число раз."""
    count = (right - left) / step
    if abs(count - round(count)) > 1e-9:
        raise ValueError(f"шаг {step} не укладывается целое число раз в [{left}, {right}]")
    return left + step * np.arange(round(count) + 1)


def left_rectangles(x: np.ndarray, step: float) -> float:
    """Левые прямоугольники: на каждом отрезке берётся значение в левом конце."""
    return float(step * sum(f(point) for point in x[:-1]))


def right_rectangles(x: np.ndarray, step: float) -> float:
    """Правые прямоугольники: значение в правом конце отрезка."""
    return float(step * sum(f(point) for point in x[1:]))


def middle_rectangles(x: np.ndarray, step: float) -> float:
    """Средние прямоугольники: значение в середине отрезка."""
    return float(step * sum(f((x[i] + x[i + 1]) / 2.0) for i in range(len(x) - 1)))


def trapezoid(x: np.ndarray, step: float) -> float:
    """Метод трапеций: полусумма значений на концах каждого отрезка."""
    values = np.array([f(point) for point in x])
    return float(step * (values[0] + values[-1] + 2.0 * values[1:-1].sum()) / 2.0)


def simpson(x: np.ndarray, step: float) -> float:
    """Метод Симпсона; требует чётного числа отрезков."""
    if (len(x) - 1) % 2 != 0:
        raise ValueError("метод Симпсона требует чётного числа отрезков")
    values = np.array([f(point) for point in x])
    return float(step * (values[0] + values[-1] + 4.0 * values[1:-1:2].sum() + 2.0 * values[2:-1:2].sum()) / 3.0)


def runge_romberg(coarse: float, fine: float, ratio: float, order: int) -> float:
    """Уточнение по Рунге-Ромбергу: I = I(h2) + (I(h2) - I(h1)) / (k^p - 1)."""
    return fine + (fine - coarse) / (ratio ** order - 1.0)


METHODS = (
    ("Левые прямоугольники", left_rectangles, 1),
    ("Правые прямоугольники", right_rectangles, 1),
    ("Средние прямоугольники", middle_rectangles, 2),
    ("Трапеции", trapezoid, 2),
    ("Симпсон", simpson, 4),
)

### Исходные данные и сетки

In [4]:
data = read_data("lab3_5.json")
left = scalar(data, "left")
right = scalar(data, "right")
h1 = scalar(data, "h1")
h2 = scalar(data, "h2")

print("Подынтегральная функция: y = 1 / (3x^2 + 4x + 2)")
print(f"Отрезок интегрирования: [{left:g}, {right:g}]")
for step in (h1, h2):
    grid = nodes_of(left, right, step)
    print(f"  шаг h = {step:g}: отрезков {len(grid) - 1}, узлов {len(grid)}")

for step in (h1, h2):
    grid = nodes_of(left, right, step)
    table([[number(point, 3), number(f(point), 9)] for point in grid], ["x_i", "y_i"],
          f"Значения подынтегральной функции при h = {step:g}")

Подынтегральная функция: y = 1 / (3x^2 + 4x + 2)
Отрезок интегрирования: [-2, 2]
  шаг h = 1: отрезков 4, узлов 5
  шаг h = 0.5: отрезков 8, узлов 9
Значения подынтегральной функции при h = 1


x_i,y_i
-2.000,0.166666667
-1.000,1.000000000
0.000,0.500000000
1.000,0.111111111
2.000,0.045454545


Значения подынтегральной функции при h = 0.5


x_i,y_i
-2.000,0.166666667
-1.500,0.363636364
-1.000,1.000000000
-0.500,1.333333333
0.000,0.500000000
0.500,0.210526316
1.000,0.111111111
1.500,0.067796610
2.000,0.045454545


### Результаты всех методов и уточнение по Рунге–Ромбергу (пункты 1-5)

In [5]:
grid1 = nodes_of(left, right, h1)
grid2 = nodes_of(left, right, h2)
ratio = h1 / h2

results = []
for title, method, order in METHODS:
    coarse = method(grid1, h1)
    fine = method(grid2, h2)
    refined = runge_romberg(coarse, fine, ratio, order)
    results.append((title, coarse, fine, order, refined, abs(refined - fine)))

table([[title, f"{coarse:.9f}", f"{fine:.9f}", order, f"{refined:.9f}", f"{estimate:.3e}"]
       for title, coarse, fine, order, refined, estimate in results],
      ["метод", f"I(h1={h1:g})", f"I(h2={h2:g})", "p", "Рунге-Ромберг",
       "оценка погрешности I(h2)"],
      "Результаты численного интегрирования")
print(f"Уточнение: I = I(h2) + (I(h2) - I(h1)) / (k^p - 1), k = h1/h2 = {ratio:g}")
print("Оценка погрешности результата с шагом h2: |I(Рунге-Ромберг) - I(h2)|")

Результаты численного интегрирования


метод,I(h1=1),I(h2=0.5),p,Рунге-Ромберг,оценка погрешности I(h2)
Левые прямоугольники,1.777777778,1.876535200,1,1.975292623,9.876e-02
Правые прямоугольники,1.656565657,1.815929140,1,1.975292623,1.594e-01
Средние прямоугольники,1.975292623,1.865925898,2,1.829470323,3.646e-02
Трапеции,1.717171717,1.846232170,2,1.889252321,4.302e-02
Симпсон,1.885521886,1.889252321,4,1.889501017,2.487e-04


Уточнение: I = I(h2) + (I(h2) - I(h1)) / (k^p - 1), k = h1/h2 = 2
Оценка погрешности результата с шагом h2: |I(Рунге-Ромберг) - I(h2)|


### Сравнение с точным значением

In [6]:
exact = antiderivative(right) - antiderivative(left)
print(f"Точное значение интеграла: I = {exact:.9f}")

table([[title, f"{fine:.9f}", f"{abs(fine - exact):.9f}", f"{refined:.9f}",
        f"{abs(refined - exact):.9f}"]
       for title, _coarse, fine, _order, refined, _estimate in results],
      ["метод", "I(h2)", "|I(h2) - I|", "Рунге-Ромберг", "|I(Р-Р) - I|"],
      "Абсолютная погрешность каждого метода")

best = min(results, key=lambda item: abs(item[2] - exact))
print(f"Наиболее точный результат с шагом h2 даёт метод: {best[0]} "
      f"(погрешность {abs(best[2] - exact):.9f}).")

Точное значение интеграла: I = 1.857418687
Абсолютная погрешность каждого метода


метод,I(h2),|I(h2) - I|,Рунге-Ромберг,|I(Р-Р) - I|
Левые прямоугольники,1.876535200,0.019116513,1.975292623,0.117873936
Правые прямоугольники,1.815929140,0.041489547,1.975292623,0.117873936
Средние прямоугольники,1.865925898,0.008507211,1.829470323,0.027948364
Трапеции,1.846232170,0.011186517,1.889252321,0.031833634
Симпсон,1.889252321,0.031833634,1.889501017,0.032082329


Наиболее точный результат с шагом h2 даёт метод: Средние прямоугольники (погрешность 0.008507211).
